<a href="https://colab.research.google.com/github/Lornatt88/maai902-nlp-solution/blob/main/notebooks/06_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 6: Demo
The Fine Print Detector as a working app: paste a Terms of Service, and it flags the unfair clauses, names their type, and explains them in plain English.

## 1. Setup
Loads the trained Legal-BERT model (run A) and the knowledge base from Google Drive.

In [2]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers gradio nltk

import torch, numpy as np, pandas as pd
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSequenceClassification

drive.mount("/content/drive")
save_dir = "/content/drive/MyDrive/fine_print_detector"
print("GPU:", torch.cuda.is_available())

kb = pd.read_csv(f"{save_dir}/knowledge_base.csv")          # the 87 explanations

clf_dir = f"{save_dir}/models/legal_bert_A"                 # trained model
clf_tok = AutoTokenizer.from_pretrained(clf_dir)
clf = AutoModelForSequenceClassification.from_pretrained(clf_dir).to("cuda").eval()
names = [clf.config.id2label[i] for i in range(8)]

print("Explanations:", len(kb))
print("Types:", names)

Mounted at /content/drive
GPU: True


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Explanations: 87
Types: ['Limitation of liability', 'Unilateral termination', 'Unilateral change', 'Content removal', 'Contract by using', 'Choice of law', 'Jurisdiction', 'Arbitration']


## 1b. Loading the LLM
Loads Qwen2.5-7B-Instruct (4-bit) on the GPU, used for the extractive summaries of long sentences and for the explanations.

In [3]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

llm_name = "Qwen/Qwen2.5-7B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)

def ask_llm(prompt, max_new_tokens=250):
    """Send one message to the LLM and return its answer."""
    messages = [{"role": "user", "content": prompt}]
    text = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return llm_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(ask_llm("Say hello in one short sentence."))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Hello!


## 2. Retrieval helpers
Loads MiniLM and the reranker, and defines retrieve(): it finds the 3 expert explanations closest to a flagged clause, within its unfair type (the final setup from Notebook 5).

In [4]:
from sentence_transformers import SentenceTransformer, CrossEncoder

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
kb_vecs = embedder.encode(kb["text"].tolist(), normalize_embeddings=True)

def retrieve(clause, clause_type, k=3, shortlist=30):
    """Shortlist explanations of this type with MiniLM, re-order them with the reranker, return the top k."""
    mask = (kb["type"] == clause_type).values
    sub = kb[mask]
    q = embedder.encode([clause], normalize_embeddings=True)[0]
    first = np.argsort(-(kb_vecs[mask] @ q))[:shortlist]
    rs = reranker.predict([(clause, sub["text"].iloc[c]) for c in first])
    order = np.argsort(-rs)[:k]
    return sub.iloc[first[order]]

print(retrieve("we may terminate your account at any time without notice .", "Unilateral termination")[["name", "severity"]])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

          name            severity
64     dormant  potentially unfair
80  any_reason      clearly unfair
68    payments  potentially unfair


## 3. Detection
Splits pasted text into sentences, removes fragments under 5 words (Lippi et al., 2019), formats each sentence like the training data, and runs Legal-BERT. A sentence is flagged for every type with a score above 0.5. Long sentences are cut at 128 tokens for now; the extractive summary is added once the LLM is loaded.

In [5]:
import re, nltk
nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

def to_dataset_style(s):
    """Format like the training data: lowercase, the dataset's quote marks, spaces around punctuation."""
    s = s.lower().strip().replace("“", '"').replace("”", '"').replace("’", "'")
    s = re.sub(r'"([^"]*)"', r" `` \1 '' ", s)
    s = re.sub(r"(\w)'s\b", r"\1 's", s)
    s = re.sub(r"\s*([,;:()])\s*", r" \1 ", s)
    s = re.sub(r"\s*\.$", " .", s)
    return re.sub(r"\s+", " ", s).strip()

def prepare(text):
    """Split into sentences and keep those with at least 5 words."""
    sentences = []
    for line in text.split("\n"):
        sentences += sent_tokenize(line)
    return [s.strip() for s in sentences if len(s.split()) >= 5]

def detect(sentences, threshold=0.5):
    """Run Legal-BERT on each sentence and return the flagged ones with their types and scores."""
    model_texts = [to_dataset_style(s) for s in sentences]
    enc = clf_tok(model_texts, truncation=True, max_length=128, padding=True, return_tensors="pt")
    with torch.no_grad():
        probs = torch.sigmoid(clf(**enc).logits).numpy()
    flagged = []
    for s, p in zip(sentences, probs):
        types = [(names[i], float(p[i])) for i in np.argsort(-p) if p[i] > threshold]
        if types:
            flagged.append({"sentence": s, "types": types})
    return flagged

sample = """Terms of Use
We may terminate your account at any time without notice.
You can contact our support team by email.
By using the app, you agree to these terms.
Any dispute will be resolved by binding arbitration."""

for f in detect(prepare(sample)):
    print(f["sentence"], "->", [(t, round(p, 2)) for t, p in f["types"]])

We may terminate your account at any time without notice. -> [('Unilateral termination', 0.99)]
By using the app, you agree to these terms. -> [('Contract by using', 0.99)]
Any dispute will be resolved by binding arbitration. -> [('Arbitration', 0.94)]
